# 05 — Clustering Analysis (Unsupervised Learning)

## What to do
Discover natural groups (clusters) in the data without predefined labels using unsupervised learning.

## What you'll learn
- K-means clustering algorithm
- Determine optimal number of clusters (Elbow method)
- Evaluate clusters using Silhouette analysis
- Visualize and interpret cluster characteristics
- Segment patient populations by health profile

## Step 1: Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, silhouette_samples
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

print('Libraries loaded!')

## Step 2: Load and prepare data

**Action:** Load dataset and prepare for clustering.
**Insight:** Clustering works on all features. We use scaled data for fair distance calculation.

In [ ]:
df = pd.read_csv('data/diabetes_prediction_dataset.csv')

# Select only numeric features for clustering
numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
X = df[numeric_cols]

# Encode categorical variables if any
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
if categorical_cols:
    X_cat = pd.get_dummies(df[categorical_cols], drop_first=True)
    X = pd.concat([X, X_cat], axis=1)

# Scale the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print(f'Data prepared for clustering: {X_scaled.shape}')

## Step 3: Elbow method (find optimal clusters)

**Action:** Test K-means with different cluster numbers and plot inertia.
**Insight:** The 'elbow' point indicates optimal clusters. Inertia decreases with more clusters.

In [ ]:
inertias = []
silhouette_scores = []
cluster_range = range(2, 11)

print('Testing cluster numbers 2-10...')
for k in cluster_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)
    silhouette = silhouette_score(X_scaled, kmeans.labels_)
    silhouette_scores.append(silhouette)
    print(f'  K={k}: Inertia={kmeans.inertia_:.2f}, Silhouette={silhouette:.4f}')

print('✓ Complete')

## Step 4: Visualize elbow curve

**Action:** Plot inertia and silhouette scores.
**Insight:** Look for the 'elbow' where inertia gain diminishes, or highest silhouette score.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow curve
axes[0].plot(cluster_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method For Optimal K')
axes[0].grid()

# Silhouette scores
axes[1].plot(cluster_range, silhouette_scores, 'go-', linewidth=2, markersize=8)
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score by K')
axes[1].grid()

plt.tight_layout()
plt.show()

# Find optimal K
optimal_k = cluster_range[np.argmax(silhouette_scores)]
print(f'\nOptimal K (by Silhouette): {optimal_k}')

## Step 5: Train final K-means model

**Action:** Train K-means with optimal cluster number.
**Insight:** Final model will be used to analyze cluster characteristics.

In [ ]:
kmeans_final = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
clusters = kmeans_final.fit_predict(X_scaled)

# Add cluster labels to original data
df['Cluster'] = clusters

print(f'K-means trained with K={optimal_k}')
print(f'\nCluster distribution:')
print(df['Cluster'].value_counts().sort_index())

## Step 6: Silhouette analysis

**Action:** Visualize silhouette plot for cluster quality.
**Insight:** Width of silhouette bars shows cluster cohesion. Wider bars = better separation.

In [ ]:
silhouette_vals = silhouette_samples(X_scaled, clusters)

plt.figure(figsize=(10, 6))
y_lower = 10

for i in range(optimal_k):
    cluster_silhouette_vals = silhouette_vals[clusters == i]
    cluster_silhouette_vals.sort()
    
    size_cluster_i = cluster_silhouette_vals.shape[0]
    y_upper = y_lower + size_cluster_i
    
    plt.fill_betweenx(np.arange(y_lower, y_upper),
                      0, cluster_silhouette_vals,
                      label=f'Cluster {i}')
    y_lower = y_upper + 10

plt.axvline(x=silhouette_score(X_scaled, clusters), color='red', linestyle='--', label='Avg Score')
plt.xlabel('Silhouette Coefficient')
plt.ylabel('Cluster Label')
plt.title('Silhouette Plot for K-Means Clusters')
plt.legend()
plt.tight_layout()
plt.show()

## Step 7: Analyze cluster characteristics

**Action:** Compute mean values for each cluster.
**Insight:** Shows what defines each cluster. Helps interpret cluster meanings.

In [ ]:
cluster_profiles = df.groupby('Cluster').mean()

print('Cluster Profiles (Mean Values):')
display(cluster_profiles.round(2))

## Step 8: Visualize clusters (2D projection)

**Action:** Use PCA to visualize high-dimensional clusters in 2D.
**Insight:** Shows cluster separation in lower dimensions for intuition.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(10, 7))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=clusters, cmap='viridis', s=50, alpha=0.6)

# Plot cluster centers
centers_pca = pca.transform(kmeans_final.cluster_centers_)
plt.scatter(centers_pca[:, 0], centers_pca[:, 1], c='red', marker='X', s=300, edgecolors='black', linewidths=2, label='Centroids')

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)')
plt.title(f'K-Means Clusters (K={optimal_k})')
plt.colorbar(scatter, label='Cluster')
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()

## Step 9: Cluster size and diabetes distribution

**Action:** Analyze diabetes prevalence in each cluster.
**Insight:** Shows which clusters have higher diabetes risk.

In [ ]:
cluster_diabetes = df.groupby('Cluster')['diabetes'].agg(['count', 'sum', 'mean'])
cluster_diabetes.columns = ['Total', 'Diabetes_Count', 'Diabetes_Rate']

print('Diabetes Distribution by Cluster:')
display(cluster_diabetes.round(4))

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cluster_diabetes['Total'].plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Cluster Sizes')
axes[0].set_xlabel('Cluster')
axes[0].set_ylabel('Count')

cluster_diabetes['Diabetes_Rate'].plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Diabetes Rate by Cluster')
axes[1].set_xlabel('Cluster')
axes[1].set_ylabel('Diabetes Rate')
axes[1].set_ylim([0, 1])

plt.tight_layout()
plt.show()

## Clustering Summary

**What we did:**
1. Loaded and scaled data
2. Applied Elbow method to find optimal K
3. Evaluated using Silhouette scores
4. Trained final K-means model
5. Analyzed cluster characteristics
6. Visualized clusters in 2D (PCA)
7. Examined diabetes distribution across clusters

**Key Insights:**
- Discovered natural patient segments
- Identified high-risk clusters
- Showed cluster separation quality

**Next:** Social media integration for extended analysis.